# UNIVERSIDADE FEDERAL DO TOCANTINS (UFT)
## Bacharelado Interdisciplinar em Inteligência Artificial (BIINAR)
### Eixo II — Projeto Integrador: Sistema de Inteligência Tributária (Sefin Palmas)
**Sprint 4 — Modelos Preditivos de IA, Scoring de Recuperabilidade e MVP**

---

- **Coordenação:** Prof. Dr. David Nadler Prata (Módulo 6 — Introdução à Inteligência Artificial)
- **Equipe Executora:** Hyag Queiroz, Ludmylla Teixeira, Maiara Xerente, Sofia Barros e Vitória Alves
- **Data:** Novembro de 2026

---

## 🔒 DECLARAÇÃO EXPLÍCITA DE CONFORMIDADE ÉTICA, LGPD E DADOS SINTÉTICOS

> **NOTA OBRIGATÓRIA DE HOMOLOGAÇÃO:**  
> Conforme exigido pela coordenação pedagógica do Eixo II e estipulado pela Lei Geral de Proteção de Dados (Lei nº 13.709/2018 - LGPD), **declara-se explicitamente que a camada individual de contribuintes utilizada nesta etapa possui natureza estritamente sintética e anonimizada**. Os dados foram calibrados estatisticamente com base nos agregados fiscais reais de Palmas-TO obtidos nas Sprints 2 e 3 (indicadores socioeconômicos do Censo IBGE por setor urbano e histórico orçamentário da Sefin), demonstrando a plena funcionalidade técnica e metodológica do pipeline preditivo sem violar a privacidade de pessoas naturais.

## 1. Importação de Bibliotecas e Configuração do Ambiente

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Ambiente carregado com sucesso!")

## 2. Ingestão e Inspeção dos Dados Tratados da Dívida Ativa

In [2]:
# Caminho dos dados prontos da Sefin Palmas
caminho_dados = os.path.join("..", "data", "divida_ativa_palmas.csv")
df = pd.read_csv(caminho_dados)
print(f"Volume de dados carregado: {df.shape[0]} registros e {df.shape[1]} colunas.")
df.head(3)

## 3. Metodologia Anti-Leakage: Partição Temporal (Entregável E1)

Para simular com fidelidade o processo contínuo de cobrança tributária e impedir o vazamento de dados de exercícios futuros (*data leakage*), foi empregada uma ordenação temporal estrita:
- **Treino:** Inscrições até o exercício de 2022.
- **Validação:** Inscrições do exercício de 2023.
- **Teste Cego:** Inscrições do exercício de 2024.

In [3]:
# Criação do Target de Recuperabilidade Efetiva (1 se Score >= 50, 0 caso contrário)
df["target_recuperavel"] = (df["score_recuperabilidade"] >= 50.0).astype(int)

num_features = [
    "tempo_inscricao_meses", "valor_principal", "juros_multas", 
    "valor_total_divida", "quantidade_debitos", "historico_parcelamentos"
]
cat_features = [
    "tributo", "tipo_pessoa", "execucao_ajuizada", 
    "possui_bens", "contato_atualizado", "faixa_renda", "regiao"
]

train_mask = df["ano_inscricao"] <= 2022
val_mask = df["ano_inscricao"] == 2023
test_mask = df["ano_inscricao"] >= 2024

X_train = df.loc[train_mask, num_features + cat_features]
y_train = df.loc[train_mask, "target_recuperavel"]

X_val = df.loc[val_mask, num_features + cat_features]
y_val = df.loc[val_mask, "target_recuperavel"]

X_test = df.loc[test_mask, num_features + cat_features]
y_test = df.loc[test_mask, "target_recuperavel"]

print(f"Partição Temporal -> Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}")

## 4. Pipeline de Pré-processamento e Treinamento dos Baselines (E1)

Construção do transformador tabular unificado com normalização z-score para numéricas e One-Hot Encoding para categóricas, seguido pelo ajuste dos dois modelos baseline:
1. **Regressão Logística**
2. **Árvore de Decisão**

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features)
    ]
)

# Baseline 1: Regressão Logística
pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

# Baseline 2: Árvore de Decisão
pipe_dt = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=6, random_state=42))
])
pipe_dt.fit(X_train, y_train)

print("Baselines ajustados com sucesso!")

## 5. Modelos Finais de Comitê de Árvores: Random Forest e XGBoost (E2)

Avanço metodológico para algoritmos ensemble baseados em árvores com regularização e calibragem de hiperparâmetros.

In [5]:
# Modelo Principal Campeão: Random Forest
pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_split=5, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)

print("Random Forest treinado com sucesso!")

## 6. Avaliação Comparativa no Conjunto de Teste Cego Temporal (E2 & E3)

Extração das métricas oficiais de Acurácia, Precisão, Recall, F1-Score e ROC-AUC sobre o exercício fiscal de 2024.

In [6]:
modelos = {
    "Regressão Logística (Baseline)": pipe_lr,
    "Árvore de Decisão (Baseline)": pipe_dt,
    "Random Forest (Campeão)": pipe_rf
}

resultados = []
for nome, mod in modelos.items():
    preds = mod.predict(X_test)
    probs = mod.predict_proba(X_test)[:, 1] if hasattr(mod, "predict_proba") else preds
    resultados.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_test, preds),
        "Precisão": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1-Score": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probs)
    })

df_res = pd.DataFrame(resultados)
df_res.style.format({
    "Acurácia": "{:.2%}",
    "Precisão": "{:.2%}",
    "Recall": "{:.2%}",
    "F1-Score": "{:.2%}",
    "ROC-AUC": "{:.4f}"
})

## 7. Matrizes de Confusão no Teste Temporal

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nome, mod) in zip(axes, modelos.items()):
    cm = confusion_matrix(y_test, mod.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Não Recuperado', 'Recuperado'],
                yticklabels=['Não Recuperado', 'Recuperado'])
    ax.set_title(nome, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predito pela IA')
    ax.set_ylabel('Real Histórico')
plt.tight_layout()
plt.show()

## 8. Motor de Scoring de Recuperabilidade e Proteção ao IPTU Social (E3)

O motor de scoring combina a probabilidade predita pelo modelo com os fatores materiais do crédito tributário (antiguidade, garantias, solvência setorial):
$$Score = 0.50 \times (100 \times P_{\text{recup}}) + 0.50 \times I_{\text{material}}$$

As faixas resultantes orientam a estratégia de cobrança da Sefin Palmas:
- **Alta ($Score \ge 68$):** Canal amigável digital (WhatsApp/DDA);
- **Média ($42 \le Score < 68$):** Notificação postal e protesto cartorário;
- **Baixa ($Score < 42$):** Ajuizamento de execução fiscal e penhora de bens.

In [8]:
# Distribuição das Faixas de Recuperabilidade na Base da Sefin Palmas
faixas_dist = df['faixa_recuperabilidade'].value_counts(normalize=True) * 100
plt.figure(figsize=(7, 3.5))
colors = ['#15803d', '#b45309', '#b91c1c']
faixas_dist.loc[['Alta', 'Média', 'Baixa']].plot(kind='bar', color=colors, edgecolor='black', alpha=0.85)
plt.title('Distribuição de Faixas de Recuperabilidade da Dívida Ativa (Sefin Palmas)', fontsize=12, fontweight='bold')
plt.ylabel('Percentual de Contribuintes (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Conclusão da Sprint 4 e Preparação para a Sprint 5

- **Objetivo Alcançado:** Os modelos treinados foram validados com split temporal e serializados com sucesso no diretório `/models`.
- **MVP Funcional:** O painel interativo em Streamlit (`app.py`) conecta-se a esses artefatos, permitindo a priorização em tempo real das cobranças.
- **Próximos Passos (Sprint 5):** Auditoria formal de equidade algorítmica (*fairness*) e elaboração do Relatório de Impacto Algorítmico (RIA) sobre os recortes socioeconômicos e territoriais já salvos.